In [5]:
import asyncio
import aiohttp
import pandas as pd
import json
import os
import base64
import re
from tqdm.asyncio import tqdm

API_KEY = "sk-824a080f44c7c8de2bd15ff4e13d386b4581ecc54ae1d506b049183d13a8c808"
API_URL = "https://ckey.vn/v1/chat/completions"
MODEL = "minimax-m2.5"

INPUT_CSV = "../data/processed/dataset_final_qwen_filled.csv"
OUTPUT_CSV = "../data/processed/dataset_metadata_filled.csv"
IMAGE_DIR = "../data/raw/images"
CHECKPOINT_FILE = "../data/processed/relabel_checkpoint.jsonl" 

BATCH_SIZE_TEXT = 20
BATCH_SIZE_VISION = 5 
CONCURRENCY_LIMIT = 15 

ENUMS = """
1. "style_aesthetic": [Minimalist/Basic, Streetwear/Y2K, Vintage/Retro, Bohemian/Resort, Sporty/Athleisure, Office/Smart Chic, Edgy/Grunge, Preppy/Academic, Glam/Party, Loungewear/Comfy, Intimate/Sensual, Unknown]
2. "dominant_material": [Cotton/Jersey, Denim/Chambray, Leather/Faux Leather, Linen/Hemp, Silk/Satin/Chiffon, Knitted/Ribbed, Wool/Fleece/Cashmere, Synthetic/Nylon/Polyester, Velvet/Corduroy, Lace/Mesh/Tulle, Unknown]
3. "design_detail": [Cropped, High-waisted, V-neck/Plunge, Turtleneck/High-neck, Sleeveless/Halter, Button-up/Polo, Zip-up, Pleated/Draped, Ruffled/Tiered, Cut-out/Open-back, Asymmetric, Graphic/Text Print, Embroidery/Applique, Sequined/Metallic, None, Unknown]
4. "fit": [Tight/Skinny/Bodycon, Slim/Tailored, Regular/Straight, Oversized/Loose/Relaxed, Flowy/Drapey, Flared/Wide-leg, Unknown]
5. "occasion": [Casual/Everyday, Office/Workwear, Party/Evening/Wedding, Sport/Active/Workout, Lounge/Sleep/Nightwear, Beach/Swimwear, Outdoor/Adventure, Intimate/Underwear, Unknown]
6. "seasonality": [Spring/Summer, Autumn/Winter, Core/All-year, Unknown]
"""

SYSTEM_PROMPT = f"""
You are an expert Fashion Data Annotator. Extract and infer attributes for fashion products.
If the provided text does not contain enough information to confidently guess an attribute, strictly use "Unknown".

CRITICAL RULES:
1. You MUST output ONLY a valid JSON object.
2. DO NOT wrap the JSON in markdown code blocks.
3. DO NOT output any greetings, explanations, or conversational text.
4. You MUST use ONLY the exact values provided in the ALLOWED ENUMS.
5. Do not assign upper-body details like 'Button-up/Polo' or 'V-neck' to bottom garments like trousers or skirts.

ALLOWED ENUMS:
{ENUMS}

Output format:
{{
  "results": [
    {{
      "article_id": "...",
      "style_aesthetic": "...",
      "dominant_material": "...",
      "design_detail": "...",
      "fit": "...",
      "occasion": "...",
      "seasonality": "..."
    }}
  ]
}}
"""
print("setup loaded")

setup loaded


In [6]:
def get_image_base64(article_id):
    folder = article_id[:3]
    img_path = os.path.join(IMAGE_DIR, folder, f"{article_id}.jpg")
    if not os.path.exists(img_path):
        return None
    with open(img_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

async def call_api(session, payload, attempt=1):
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    try:
        async with session.post(API_URL, headers=headers, json=payload, timeout=90) as response:
            if response.status == 200:
                data = await response.json()
                content = data['choices'][0]['message']['content']
                
                json_match = re.search(r'\{.*\}', content, re.DOTALL)
                
                if json_match:
                    clean_json = json_match.group(0)
                    try:
                        return json.loads(clean_json).get("results", [])
                    except json.JSONDecodeError as e:
                        print(f"parse error: {e}")
                        return []
                else:
                    print("format error")
                    return []
                
            elif response.status == 400:
                print("fatal 400 error")
                return []
            elif response.status in [502, 504]:
                if attempt < 3:
                    await asyncio.sleep(2 ** attempt)
                    return await call_api(session, payload, attempt + 1)
                return []
            else:
                if attempt < 3:
                    await asyncio.sleep(2 ** attempt)
                    return await call_api(session, payload, attempt + 1)
                return []
    except Exception:
        if attempt < 3:
            await asyncio.sleep(2 ** attempt)
            return await call_api(session, payload, attempt + 1)
        return []

async def process_batch(session, batch_data, semaphore, use_vision=False):
    async with semaphore:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        
        text_context = []
        for item in batch_data:
            text_context.append({
                "article_id": item['article_id'],
                "category_context": f"{item['index_name']} > {item['product_group_name']} > {item['department_name']} > {item['product_type_name']}",
                "appearance": item['graphical_appearance'],
                "detail_desc": item['detail_desc'],
                "refined_description": item['refined_description']
            })
            
        if not use_vision:
            user_content = json.dumps(text_context, ensure_ascii=False)
            messages.append({"role": "user", "content": user_content})
        else:
            user_blocks = [{"type": "text", "text": json.dumps(text_context, ensure_ascii=False)}]
            for item in batch_data:
                img_b64 = get_image_base64(item['article_id'])
                if img_b64:
                    user_blocks.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}", "detail": "low"}})
            messages.append({"role": "user", "content": user_blocks})
            
        payload = {"model": MODEL, "messages": messages, "temperature": 0.1}
        return await call_api(session, payload)

print("functions loaded")

functions loaded


In [7]:
df = pd.read_csv(INPUT_CSV, dtype={'article_id': str})

processed_results = {}
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            res = json.loads(line)
            processed_results[res["article_id"]] = res

items_to_process = []
for _, row in df.iterrows():
    aid = str(row['article_id']).zfill(10)
    if aid not in processed_results:
        items_to_process.append({
            "article_id": aid,
            "index_name": str(row.get('index_name', '')),
            "product_group_name": str(row.get('product_group_name', '')),
            "department_name": str(row.get('department_name', '')),
            "product_type_name": str(row.get('product_type_name', '')),
            "graphical_appearance": str(row.get('graphical_appearance_name', '')),
            "detail_desc": str(row.get('detail_desc', '')),
            "refined_description": str(row.get('refined_description', ''))
})
if items_to_process:
    batches = [items_to_process[i:i + BATCH_SIZE_TEXT] for i in range(0, len(items_to_process), BATCH_SIZE_TEXT)]
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    
    async with aiohttp.ClientSession() as session:
        tasks = [process_batch(session, batch, semaphore, use_vision=False) for batch in batches]
        
        with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f_out:
            for f in tqdm(asyncio.as_completed(tasks), total=len(batches)):
                batch_result = await f
                for item_res in batch_result:
                    f_out.write(json.dumps(item_res, ensure_ascii=False) + "\n")
                    processed_results[item_res["article_id"]] = item_res

print("text pass complete")

  0%|          | 15/3510 [07:14<28:07:37, 28.97s/it] 


CancelledError: 

In [ ]:
items_for_vision = []
for aid, res in processed_results.items():
    if any(val == "Unknown" for val in res.values()):
        row = df[df['article_id'] == aid].iloc[0]
        items_for_vision.append({
            "article_id": aid,
            "index_name": str(row.get('index_name', '')),
            "product_group_name": str(row.get('product_group_name', '')),
            "department_name": str(row.get('department_name', '')),
            "product_type_name": str(row.get('product_type_name', '')),
            "graphical_appearance": str(row.get('graphical_appearance_name', '')),
            "detail_desc": str(row.get('detail_desc', '')),
            "refined_description": str(row.get('refined_description', ''))
})
if items_for_vision:
    batches_v = [items_for_vision[i:i + BATCH_SIZE_VISION] for i in range(0, len(items_for_vision), BATCH_SIZE_VISION)]
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    
    async with aiohttp.ClientSession() as session:
        tasks_v = [process_batch(session, batch, semaphore, use_vision=True) for batch in batches_v]
        
        for f in tqdm(asyncio.as_completed(tasks_v), total=len(batches_v)):
            batch_result = await f
            for item_res in batch_result:
                aid = item_res.get("article_id")
                if aid in processed_results:
                    processed_results[aid].update(item_res)

df_new = pd.DataFrame(list(processed_results.values()))
df_new = df_new.rename(columns={'fit': 'fit_llm', 'occasion': 'occasion_llm', 'seasonality': 'seasonality_llm'})

df_final = df.merge(df_new, on="article_id", how="left")
df_final['fit'] = df_final['fit_llm'].combine_first(df_final['fit'])
df_final['occasion'] = df_final['occasion_llm'].combine_first(df_final['occasion'])
df_final['seasonality'] = df_final['seasonality_llm'].combine_first(df_final['seasonality'])
df_final = df_final.drop(columns=['fit_llm', 'occasion_llm', 'seasonality_llm'])

df_final.to_csv(OUTPUT_CSV, index=False)
print("pipeline finished")